# Create an Invoice

Use this notebook to validate `data/invoice.json`, review the calculated amounts, and create a PDF in `output/`. Update the invoice JSON first, then run all cells in order.

## Configuration

The package defines the repository root, so these paths work whether VS Code opens the notebook from the project root or the `notebooks/` directory.

In [ ]:
import os
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display
from pydantic import ValidationError

from invoice_generator import (
    create_receipt,
    format_currency,
    generate_invoice,
    prepare_invoice,
    record_full_payment,
)
from invoice_generator.loader import PROJECT_ROOT

BUSINESS_PATH = Path(
    os.environ.get("INVOICE_BUSINESS_PATH", PROJECT_ROOT / "data" / "business.json")
)
INVOICE_PATH = Path(os.environ.get("INVOICE_INVOICE_PATH", PROJECT_ROOT / "data" / "invoice.json"))
OUTPUT_DIR = Path(os.environ.get("INVOICE_OUTPUT_DIR", PROJECT_ROOT / "output"))
RECEIPT_DIR = Path(os.environ.get("INVOICE_RECEIPT_DIR", PROJECT_ROOT / "data" / "receipts"))
PAYMENT_RECORD_DIR = Path(
    os.environ.get("INVOICE_PAYMENT_RECORD_DIR", PROJECT_ROOT / "data" / "payment_records")
)

# Set one operation: summary, create_invoice, record_payment, or create_receipt.
OPERATION = os.environ.get("INVOICE_OPERATION", "create_invoice")
PAYMENT_AMOUNT_RECEIVED = os.environ.get("INVOICE_PAYMENT_AMOUNT", "")
PAYMENT_RECEIVED_DATE = os.environ.get("INVOICE_PAYMENT_DATE", "")
PAYMENT_REFERENCE = os.environ.get("INVOICE_PAYMENT_REFERENCE")

print(f"Business configuration: {BUSINESS_PATH}")
print(f"Invoice configuration: {INVOICE_PATH}")
print(f"PDF output directory: {OUTPUT_DIR}")
print(f"Receipt directory: {RECEIPT_DIR}")
print(f"Payment record directory: {PAYMENT_RECORD_DIR}")

## Load and Validate JSON

Both configuration files are required. This cell stops if a file is missing or the JSON does not satisfy the invoice schema. For a new invoice, omit `due_date` and the shared workflow calculates it as two calendar days after `issue_date`; an existing explicit due date is preserved.

In [ ]:
try:
    prepared = prepare_invoice(BUSINESS_PATH, INVOICE_PATH)
except FileNotFoundError as error:
    display(Markdown(f"**Configuration file missing.** {error}"))
    raise
except (ValidationError, ValueError) as error:
    display(Markdown(f"**Configuration is invalid.**\n\n```text\n{error}\n```"))
    raise

display(Markdown(f"Validated invoice **{prepared.invoice.invoice_number}**."))

## Invoice Calculations

The values below come from the existing Decimal-based calculation engine.

In [ ]:
invoice = prepared.invoice
calculation = prepared.calculation
currency = prepared.business.currency

rows = "".join(
    "<tr>"
    f"<td>{escape(line.item.description)}</td>"
    f'<td style="text-align:right">{line.item.quantity}</td>'
    f'<td style="text-align:right">{format_currency(line.item.unit_price, currency)}</td>'
    f'<td style="text-align:right">{format_currency(line.total, currency)}</td>'
    "</tr>"
    for line in calculation.line_items
)
discount_row = ""
if invoice.discount is not None:
    discount_total = format_currency(calculation.discount_total, currency)
    discount_row = (
        '<tr><td colspan="3" style="text-align:right">Discount</td>'
        f'<td style="text-align:right">-{discount_total}</td>'
        "</tr>"
    )

subtotal = format_currency(calculation.subtotal, currency)
total_payable = format_currency(calculation.total_payable, currency)
summary = (
    '<table style="min-width:620px;border-collapse:collapse">'
    "<thead><tr>"
    '<th style="text-align:left">Description</th>'
    '<th style="text-align:right">Qty</th>'
    '<th style="text-align:right">Rate</th>'
    '<th style="text-align:right">Amount</th>'
    "</tr></thead>"
    f"<tbody>{rows}</tbody>"
    "<tfoot>"
    '<tr><td colspan="3" style="text-align:right">Subtotal</td>'
    f'<td style="text-align:right">{subtotal}</td></tr>'
    f"{discount_row}"
    '<tr><th colspan="3" style="text-align:right">Total payable</th>'
    f'<th style="text-align:right">{total_payable}</th></tr>'
    "</tfoot></table>"
)
display(HTML(summary))

## Run an Operation

- `create_invoice` creates the invoice PDF and an immutable private payment snapshot.
- `record_payment` records a manually confirmed full payment only when the exact invoice amount, received date, and optional reference are supplied.
- `create_receipt` creates or recovers the one receipt permanently assigned to that payment.
- `summary` changes nothing.

The renderer uses the approved numerical layout. Production generation disables prototype-only labels. Existing invoice PDFs are never overwritten.

In [ ]:
if OPERATION == "summary":
    display(Markdown("No changes made. Select an operation when ready."))
elif OPERATION == "create_invoice":
    pdf_path = generate_invoice(
        prepared,
        OUTPUT_DIR,
        prototype=False,
        payment_record_dir=PAYMENT_RECORD_DIR,
    )
    display(Markdown(f"**Invoice created.**  \nOutput: `{pdf_path}`"))
elif OPERATION == "record_payment":
    if not all([PAYMENT_AMOUNT_RECEIVED, PAYMENT_RECEIVED_DATE]):
        raise ValueError("Provide the received amount and received date.")
    record_full_payment(
        PAYMENT_RECORD_DIR,
        prepared.invoice.invoice_number,
        PAYMENT_AMOUNT_RECEIVED,
        PAYMENT_RECEIVED_DATE,
        PAYMENT_REFERENCE,
    )
    display(Markdown("**Full payment recorded as confirmed.** No PDF was created."))
elif OPERATION == "create_receipt":
    receipt_path = create_receipt(
        PAYMENT_RECORD_DIR, prepared.invoice.invoice_number, RECEIPT_DIR
    )
    display(Markdown(f"**Receipt created.**  \nOutput: `{receipt_path}`"))
else:
    raise ValueError(
        "OPERATION must be summary, create_invoice, record_payment, or create_receipt."
    )